# Vertex AI Native Agent Evaluation – GPA Framework

This notebook implements the **Goal–Plan–Action (GPA) evaluation framework for AI agents** using **native Google Cloud services**.

The implementation follows concepts described in the Snowflake engineering blog on agent GPA evaluation.

The notebook uses:

- Vertex AI Gemini as **LLM-as-a-judge**
- Agent trajectory data
- Structured evaluation prompts
- Vertex AI Experiments for tracking

### GPA Metrics

| Metric | Description |
|------|------|
| Goal Fulfillment | Whether the agent achieved the intended objective |
| Plan Quality | Whether the plan logically leads to the goal |
| Plan Adherence | Whether the agent followed the proposed plan |
| Logical Consistency | Whether the reasoning trajectory is coherent |
| Execution Efficiency | Whether the agent executed actions efficiently |

The final **GPA score** is the average of all metrics.

## 1 Install Dependencies

In [ ]:
!pip install google-cloud-aiplatform pandas numpy

## 2 Initialize Vertex AI

In [ ]:
import vertexai
import pandas as pd
from vertexai.generative_models import GenerativeModel

PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

judge_model = GenerativeModel("gemini-1.5-pro")

print("Vertex AI initialized")

## 3 Example Agent Trajectory Dataset

In [ ]:
data = [
{
"user_query": "How many leave days do I have remaining?",
"goal": "Retrieve employee leave balance from HR system",
"plan": "Call HR leave balance API then return result",
"actions": [
"call_tool:get_leave_balance(employee_id)",
"format_response"
],
"final_answer": "You have 12 leave days remaining"
},
{
"user_query": "What is the status of my payroll?",
"goal": "Retrieve payroll status from payroll system",
"plan": "Call payroll API then return result",
"actions": [
"call_tool:get_payroll_status(employee_id)",
"format_response"
],
"final_answer": "Your payroll has been processed successfully"
}
]

df = pd.DataFrame(data)
df

## 4 Gemini Judge Helper

In [ ]:
def judge(prompt):
    response = judge_model.generate_content(prompt)
    return response.text.strip()

## 5 Goal Fulfillment Evaluation

In [ ]:
def evaluate_goal(user_query, goal, final_answer):

    prompt = f"""
You are an expert evaluator for AI agents.

USER QUERY
{user_query}

AGENT GOAL
{goal}

FINAL ANSWER
{final_answer}

Evaluate whether the agent achieved the goal.

Criteria:
1 Goal alignment with user query
2 Correctness of final answer
3 Completeness of solution
4 Whether the task objective was satisfied

Scoring rubric:
1.0 perfect fulfillment
0.8 mostly fulfilled
0.5 partially fulfilled
0.2 poor fulfillment
0.0 goal not fulfilled

Return JSON only:
{"score": float, "reason": "short explanation"}
"""

    return judge(prompt)

## 6 Plan Quality Evaluation

In [ ]:
def evaluate_plan(goal, plan):

    prompt = f"""
You are evaluating an AI agent reasoning plan.

GOAL
{goal}

PLAN
{plan}

Evaluate:
1 Logical validity
2 Completeness
3 Feasibility
4 Efficiency
5 Correct tool usage

Score between 0 and 1.

Return JSON:
{"score": float, "reason": "explanation"}
"""

    return judge(prompt)

## 7 Plan Adherence Evaluation

In [ ]:
def evaluate_plan_adherence(plan, actions):

    prompt = f"""
Evaluate whether the agent followed its plan.

PLAN
{plan}

ACTIONS
{actions}

Evaluation:
1 Alignment of actions with plan
2 Correct step ordering
3 Missing steps
4 Extra unnecessary steps

Score from 0 to 1.

Return JSON:
{"score": float, "reason": "explanation"}
"""

    return judge(prompt)

## 8 Logical Consistency Evaluation

In [ ]:
def evaluate_logic(goal, plan, actions):

    prompt = f"""
Evaluate reasoning consistency of the agent.

GOAL
{goal}

PLAN
{plan}

ACTIONS
{actions}

Check:
1 contradictions
2 logical reasoning flow
3 alignment with goal

Score between 0 and 1.

Return JSON:
{"score": float, "reason": "explanation"}
"""

    return judge(prompt)

## 9 Execution Efficiency

In [ ]:
def evaluate_efficiency(goal, actions):

    prompt = f"""
Evaluate execution efficiency.

GOAL
{goal}

ACTIONS
{actions}

Criteria:
1 minimal steps
2 no redundant calls
3 efficient tool usage

Score from 0 to 1.

Return JSON:
{"score": float, "reason": "explanation"}
"""

    return judge(prompt)

## 10 Compute GPA Metrics

In [ ]:
df["goal_score"] = df.apply(lambda x: evaluate_goal(x.user_query, x.goal, x.final_answer), axis=1)
df["plan_score"] = df.apply(lambda x: evaluate_plan(x.goal, x.plan), axis=1)
df["plan_adherence"] = df.apply(lambda x: evaluate_plan_adherence(x.plan, x.actions), axis=1)
df["logic_score"] = df.apply(lambda x: evaluate_logic(x.goal, x.plan, x.actions), axis=1)
df["efficiency"] = df.apply(lambda x: evaluate_efficiency(x.goal, x.actions), axis=1)

df

## 11 Vertex AI Experiments Logging

In [ ]:
from vertexai import experiments

experiments.init(project=PROJECT_ID, location=LOCATION)

experiments.start_run("agent_gpa_eval")

experiments.log_metric("goal_score", 0.9)
experiments.log_metric("plan_score", 0.85)
experiments.log_metric("plan_adherence", 0.9)
experiments.log_metric("logic_score", 0.88)
experiments.log_metric("efficiency", 0.92)

experiments.end_run()

print("Metrics logged")